<a href="https://colab.research.google.com/github/Atif-04/atifkhaleel_augustbatch/blob/main/RAG_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [28]:
!pip install pypdf sentence-transformers google-generativeai numpy scikit-learn -q


In [38]:
!pip install -U google-genai -q

from google.colab import userdata
from google import genai

client_genai = genai.Client(api_key=userdata.get('GOOGLE_API_KEY'))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.5/56.5 kB 1.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.7/259.7 kB 2.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.57.0 which is incompatible.


In [30]:
from google.colab import files
from pypdf import PdfReader

uploaded = files.upload()
pdf_path = list(uploaded.keys())[0]

reader = PdfReader(pdf_path)
full_text = ""
for page in reader.pages:
    full_text += page.extract_text() + "\n"

print("Total characters extracted:", len(full_text))

Saving college_cafeteria_rules.pdf to college_cafeteria_rules (4).pdf
Total characters extracted: 2512


In [31]:
def chunk_text(text, chunk_size=800, overlap=150):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

chunks = chunk_text(full_text, chunk_size=800, overlap=150)
print("Number of chunks:", len(chunks))
print("\n--- Sample chunk (chunk 2) ---\n")
print(chunks[1])

Number of chunks: 4

--- Sample chunk (chunk 2) ---

 30.
4. Lunch
Lunch is served from 12:30 PM to 3:00 PM. A standard lunch includes rice or two chapatis, one vegetable
dish, dal, curd, and a salad. Students may request a vegetarian meal without additional charge.
5. Special Meals
A special meal is available every Wednesday for Rs. 120. The Wednesday special menu is announced on the
cafeteria notice board by Monday afternoon.
6. Payment Rules
The cafeteria accepts cash, UPI, and college meal cards. Credit purchases are not allowed. Students must
keep their payment receipt until they leave the cafeteria.
7. Cleanliness Rules
Students must return used plates, cups, and cutlery to the collection counter. Food waste must be placed in
the marked bins. Throwing food or waste on tables or floors is prohibited.
8. Seating Rules
A student may occup


In [33]:
import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

!pip install -U sentence-transformers huggingface_hub -q

from sentence_transformers import SentenceTransformer
import numpy as np

embed_model = SentenceTransformer("all-MiniLM-L6-v2")

def get_embedding(text):
    return embed_model.encode(text)

chunk_embeddings = embed_model.encode(chunks)
print("Embeddings shape:", chunk_embeddings.shape)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 739.8/739.8 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 795.8/795.8 kB 17.8 MB/s eta 0:00:00


model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Embeddings shape: (4, 384)


In [34]:
from sklearn.metrics.pairwise import cosine_similarity

def search_chunks(query, top_k=3):
    query_emb = get_embedding(query).reshape(1, -1)
    scores = cosine_similarity(query_emb, chunk_embeddings)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [(chunks[i], scores[i]) for i in top_indices]

In [35]:
test_query = "How much does a lost meal card replacement cost?"
results = search_chunks(test_query, top_k=3)

for i, (chunk, score) in enumerate(results):
    print(f"--- Result {i+1} (score: {score:.4f}) ---")
    print(chunk)
    print()

--- Result 1 (score: 0.5097) ---
 reported to the cafeteria supervisor on the same day. Refund requests for
incorrect or missing items must be submitted within 24 hours of purchase.

11. Contact
The cafeteria supervisor is available at the main counter from 10:00 AM to 4:00 PM on working days.
Students can submit written suggestions in the feedback box near the entrance.
Quick Reference
 Item
Information
Monthly meal card
Rs. 2,500
Lost card replacement
Rs. 100
Wednesday special meal
Rs. 120
Extra breakfast serving
Rs. 30
Peak lunch table time
30 minutes
Refund request deadline
24 hours



--- Result 2 (score: 0.3526) ---
Greenfield College Cafeteria Rules & Information
1. Cafeteria Timings
The college cafeteria is open from 8:00 AM to 6:00 PM from Monday to Friday. On Saturdays, it operates
from 9:00 AM to 2:00 PM. The cafeteria is closed on Sundays and public holidays.
2. Student Meal Card
Students can purchase a monthly meal card for Rs. 2,500. The meal card is valid for one calenda

In [39]:
def answer_question(query, top_k=3):
    results = search_chunks(query, top_k=top_k)
    context = "\n\n".join([r[0] for r in results])

    prompt = f"""Answer the question using ONLY the context below.
If the answer is not present in the context, say "I don't know based on the provided document."
Do not use any outside knowledge.

Context:
{context}

Question: {query}

Answer:"""

    response = client_genai.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=prompt
    )
    answer = response.text

    print("QUESTION:", query)
    print("ANSWER:", answer)
    print("\nRetrieved chunks used:")
    for i, (chunk, score) in enumerate(results):
        print(f"  Chunk {i+1} (score: {score:.4f}): {chunk[:100]}...")
    print("\n" + "="*80 + "\n")

    return answer

In [40]:
answer_question("How much does the monthly meal card cost?")
answer_question("What time does lunch service end?")
answer_question("How long does a student have to submit a refund request?")

answer_question("What is the cafeteria's policy on outside food delivery?")
answer_question("Does the cafeteria offer vegan protein powder supplements?")
answer_question("What is the phone number of the cafeteria supervisor?")

QUESTION: How much does the monthly meal card cost?
ANSWER: The monthly meal card costs Rs. 2,500.

Retrieved chunks used:
  Chunk 1 (score: 0.4304): Greenfield College Cafeteria Rules & Information
1. Cafeteria Timings
The college cafeteria is open ...
  Chunk 2 (score: 0.3882):  30.
4. Lunch
Lunch is served from 12:30 PM to 3:00 PM. A standard lunch includes rice or two chapat...
  Chunk 3 (score: 0.3465):  reported to the cafeteria supervisor on the same day. Refund requests for
incorrect or missing item...


QUESTION: What time does lunch service end?
ANSWER: Lunch service ends at 3:00 PM.

Retrieved chunks used:
  Chunk 1 (score: 0.5761):  30.
4. Lunch
Lunch is served from 12:30 PM to 3:00 PM. A standard lunch includes rice or two chapat...
  Chunk 2 (score: 0.5407): Greenfield College Cafeteria Rules & Information
1. Cafeteria Timings
The college cafeteria is open ...
  Chunk 3 (score: 0.5307):  reported to the cafeteria supervisor on the same day. Refund requests for
incorrect o

"I don't know based on the provided document."